In [0]:
books_df = spark.table("workspace.default.books")

In [0]:
#parse genres column into array column
from pyspark.sql import functions as F

dim_books_df = books_df.withColumn("genres_array", F.split(F.regexp_replace(F.col("genres"), r"[\[\]']", ""), r",\s*"))

In [0]:
dim_books_df = dim_books_df.withColumn("authors_array", F.split(F.regexp_replace(F.col("authors"), r"[\[\]']", ""),r",\s*"))

In [0]:
#cast numeric columns to proper numbers
dim_books_df = dim_books_df.withColumn("original_publication_year", F.col("original_publication_year").cast("int")).withColumn("pages", F.col("pages").cast("int"))

In [0]:
from pyspark.sql.window import Window

#ntile splits all books into 4 equal size groups based on rating_count

window_spec = Window.orderBy(F.col("ratings_count"))
dim_books_df = dim_books_df.withColumn("popularity_bucket", F.ntile(4).over(window_spec))

In [0]:
dim_books_df = dim_books_df.select("book_id", "title",
    F.col("authors_array").alias("authors"),
    F.col("genres_array").alias("genres"),
    "original_publication_year", "pages","average_rating", "ratings_count", "popularity_bucket")

In [0]:
def quality_gate(df, table_name, key_column, min_expected_rows=1):
    row_count = df.count()
    null_keys = df.filter(df[key_column].isNull()).count()
    duplicate_keys = row_count - df.dropDuplicates([key_column]).count()
    if row_count < min_expected_rows:
        raise ValueError(f"[{table_name}] Row count {row_count} below minimum {min_expected_rows} — aborting.")
    if null_keys > 0:
        raise ValueError(f"[{table_name}] Found {null_keys} null values in key column '{key_column}' — aborting.")
    if duplicate_keys > 0:
        df = df.dropDuplicates([key_column])
    return df

dim_books_df = quality_gate(dim_books_df, "dim_books", key_column="book_id", min_expected_rows=9000)

In [0]:
from delta.tables import DeltaTable

def upsert_delta(spark, source_df, target_table_name, merge_key):
    if not spark.catalog.tableExists(target_table_name):
        source_df.write.format("delta").saveAsTable(target_table_name)
        return
    target = DeltaTable.forName(spark, target_table_name)
    (target.alias("t").merge(source_df.alias("s"), f"t.{merge_key} = s.{merge_key}")
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

upsert_delta(spark, dim_books_df, "workspace.default.dim_books", merge_key="book_id")

In [0]:
display(spark.table("workspace.default.dim_books").limit(5))

In [0]:
ratings_df = spark.table("workspace.default.ratings")

In [0]:
#find ratings whose book_id doesn't exist in dim_books - left join
orphaned_ratings = ratings_df.join(dim_books_df, on="book_id", how="left_anti")
orphan_count = orphaned_ratings.count()

print(f"Orphaned ratings (book_id not found in dim_books): {orphan_count}")

In [0]:
#there are no orpaned ratings

fact_ratings_df = quality_gate(ratings_df, "fact_ratings", key_column="book_id", min_expected_rows=900000)
upsert_delta(spark, fact_ratings_df, "workspace.default.fact_ratings", merge_key="book_id")

In [0]:
# uniqueness is user_id + book_id together

In [0]:
%sql
--answering question 1 from day 1 buysiness questions
--Which genres have the highest average rating vs. highest volume?

select genre, round(avg(average_rating)) as avg_rating, count(*) as num_books
from workspace.default.dim_books lateral view explode(genres) as genre
group by genre 
order by avg_rating desc
limit 15;

In [0]:
%sql
-- Which authors are most "polarizing" (min 20 ratings, highest rating variance)? 
-- polarizing -> people strongly disagree with each other
select author, round(stddev(f.rating), 2) as rating_stddev, count(*) as num_books
from workspace.default.fact_ratings f join workspace.default.dim_books d on f.book_id = d.book_id
lateral view explode(d.authors) as author
group by author
having count(*) >= 20
order by rating_stddev desc
limit 15;


In [0]:
%sql
--Does page count correlate with rating?

select count(*) as num_books, corr(pages, average_rating) as page_rating_corr
from workspace.default.dim_books
where pages is not null and average_rating is not null;

-- result is close to 0, is more like a no relationship between page and ratig 
-- page count doesn't predict rating 

In [0]:
%sql
--How does the ratings distribution trend by publication year?
select original_publication_year, round(avg(average_rating)) as avg_rating, count(*) as num_books, SUM(ratings_count) AS total_ratings
from workspace.default.dim_books
where original_publication_year is not null and original_publication_year between 1900 and 2020
group by original_publication_year
order by original_publication_year asc
;

In [0]:
%sql

-- Which books are most "underrated" (high rating, low ratings_count)?
select title, average_rating, ratings_count
from workspace.default.dim_books
where average_rating >= 4.2
  and ratings_count < 10000
  and ratings_count > 100
order by  average_rating desc, ratings_count asc
limit 15;